# DevData-01: Example Analysis & Exploration

This notebook demonstrates standard SQL-style queries and pandas workflows across the 6 relational tables of **DevData-01** as specified in `DevData-01_Dataset_Specification.md`.

### Queries covered:
- **PS-01 & PS-02**: Relational Joins (`organizations` → `subscriptions` → `invoices`)
- **PS-03**: Subscription & Billing Trends (MRR estimation & collection rate)
- **PS-04**: User Activity & Event Funnel
- **PS-05**: Support Ticket Metrics & Organization Context

In [1]:
import pandas as pd
import os

DATA_DIR = '../data'

orgs = pd.read_csv(os.path.join(DATA_DIR, 'organizations.csv'))
users = pd.read_csv(os.path.join(DATA_DIR, 'users.csv'))
subs = pd.read_csv(os.path.join(DATA_DIR, 'subscriptions.csv'))
invs = pd.read_csv(os.path.join(DATA_DIR, 'invoices.csv'))
logs = pd.read_csv(os.path.join(DATA_DIR, 'activity_logs.csv'))
tickets = pd.read_csv(os.path.join(DATA_DIR, 'support_tickets.csv'))

print(f"Successfully loaded all 6 tables ({len(orgs) + len(users) + len(subs) + len(invs) + len(logs) + len(tickets):,} total rows).")

Successfully loaded all 6 tables (19,800 total rows).


## 1. Organizations & Active Subscriptions

In [1]:
# Join organizations and active subscriptions
active_subs = subs[subs['status'] == 'active']
org_subs = active_subs.merge(orgs, on='organization_id')

plan_dist = org_subs['plan'].value_counts()
print("Active Subscriptions by Plan Tier:")
print(plan_dist)

Active Subscriptions by Plan Tier:
plan
pro           63
starter       44
enterprise    37
free           6
Name: count, dtype: int64


## 2. Invoicing & Collection Analysis (PS-03)

In [1]:
# Invoices are in amount_minor (cents)
inv_summary = invs.groupby('status').agg(
    invoice_count=('invoice_id', 'count'),
    total_billed_cents=('amount_minor', 'sum')
).reset_index()

inv_summary['total_billed_usd'] = inv_summary['total_billed_cents'] / 100.0

paid_invs = invs[invs['status'] == 'paid']
collection_rate = (paid_invs['amount_minor'].sum() / invs['amount_minor'].sum()) * 100

print(f"Overall Payment Collection Rate: {collection_rate:.2f}%")
print("\nInvoice Summary by Status:")
print(inv_summary[['status', 'invoice_count', 'total_billed_usd']])

Overall Payment Collection Rate: 91.23%

Invoice Summary by Status:
          status  invoice_count  total_billed_usd
0          draft             14            1696.0
1           open             54           13493.0
2           paid           1397          224892.0
3  uncollectible             16            1214.0
4           void             19            5212.0


## 3. Product Activity Logs (PS-04)

In [1]:
event_summary = logs['event_type'].value_counts()
print("Event Frequency Distribution:")
print(event_summary)

Event Frequency Distribution:
event_type
login                  5052
dashboard_viewed       4215
report_viewed          2534
project_created        1527
export_completed       1080
settings_updated        173
api_key_generated       153
integration_enabled     138
team_invite_sent        128
Name: count, dtype: int64


## 4. Support Operations & SLA Resolution (PS-05)

In [1]:
tickets['created_at'] = pd.to_datetime(tickets['created_at'], utc=True)
tickets['resolved_at'] = pd.to_datetime(tickets['resolved_at'], utc=True)

resolved = tickets[tickets['status'].isin(['resolved', 'closed'])].copy()
resolved['resolution_hours'] = (resolved['resolved_at'] - resolved['created_at']).dt.total_seconds() / 3600.0

print(f"Average Resolution Time: {resolved['resolution_hours'].mean():.2f} hours")
print(f"Median Resolution Time: {resolved['resolution_hours'].median():.2f} hours")

print("\nAverage Resolution Time by Priority (hours):")
print(resolved.groupby('priority')['resolution_hours'].mean().round(2))

Average Resolution Time: 25.05 hours
Median Resolution Time: 21.87 hours

Average Resolution Time by Priority (hours):
priority
high      12.83
low       41.32
medium    26.83
urgent     4.03
Name: resolution_hours, dtype: float64
